# NB_OTT_EvaluacionResultados

Objetivo del notebook:
Verificar que la refactorización de notebooks no ha alterado funcionalmente el resultado del pipeline. Usamos este notebook como test para asegurarnos que todo sigue funcionando después de realizar cambios en nuestro pipeline

In [1]:


from pyspark.sql import functions as F

StatementMeta(, d19e2b24-aee5-42e5-8f75-682471521efb, 5, Finished, Available, Finished, False)

In [2]:
# ============================================================
# 1. Carga de tablas finales
# ============================================================

df_silver = spark.table("Silver.Tickets")
df_embeddings = spark.table("Silver.TicketEmbeddings")
df_clusters = spark.table("Gold.TicketClusters")
df_clustered = spark.table("Gold.ClusteredTickets")
df_groups = spark.table("Gold.IncidentGroups")
df_summaries = spark.table("Gold.IncidentSummaries")

StatementMeta(, d19e2b24-aee5-42e5-8f75-682471521efb, 6, Finished, Available, Finished, False)

In [3]:
# ============================================================
# 2. Validación de volúmenes
# ============================================================

silver_count = df_silver.count()
embedding_count = df_embeddings.count()
cluster_count = df_clusters.count()
clustered_count = df_clustered.count()
group_count = df_groups.count()
summary_count = df_summaries.count()

print("Pipeline result counts")
print("----------------------")
print("Silver.Tickets:", silver_count)
print("Silver.TicketEmbeddings:", embedding_count)
print("Gold.TicketClusters:", cluster_count)
print("Gold.ClusteredTickets:", clustered_count)
print("Gold.IncidentGroups:", group_count)
print("Gold.IncidentSummaries:", summary_count)

StatementMeta(, d19e2b24-aee5-42e5-8f75-682471521efb, 7, Finished, Available, Finished, False)

Pipeline result counts
----------------------
Silver.Tickets: 1000
Silver.TicketEmbeddings: 1000
Gold.TicketClusters: 1000
Gold.ClusteredTickets: 1000
Gold.IncidentGroups: 14
Gold.IncidentSummaries: 14


In [4]:
# ============================================================
# 3. Validación de ingesta batch + Kafka
# ============================================================

source_counts = {
    row["ingestion_source"]: row["count"]
    for row in (
        df_silver
        .groupBy("ingestion_source")
        .count()
        .collect()
    )
}

print(source_counts)

StatementMeta(, d19e2b24-aee5-42e5-8f75-682471521efb, 8, Finished, Available, Finished, False)

{'kafka': 5, 'batch': 995}


In [5]:
# ============================================================
# 4. Validación del clustering final
# ============================================================

real_cluster_count = (
    df_clusters
    .filter(F.col("cluster_id") != -1)
    .select("cluster_id")
    .distinct()
    .count()
)

noise_count = (
    df_clusters
    .filter(F.col("cluster_id") == -1)
    .count()
)

print("Real clusters:", real_cluster_count)
print("Noise tickets:", noise_count)

assert real_cluster_count == 14
assert noise_count == 0

StatementMeta(, d19e2b24-aee5-42e5-8f75-682471521efb, 9, Finished, Available, Finished, False)

Real clusters: 14
Noise tickets: 0


In [6]:
# ============================================================
# 5. Validación de configuración
# ============================================================

configs = (
    df_clusters
    .select(
        "clustering_algorithm",
        "eps",
        "min_samples",
        "embedding_model"
    )
    .distinct()
)

display(configs)

config_rows = configs.collect()

assert len(config_rows) == 1

config = config_rows[0]

assert config["clustering_algorithm"] == "DBSCAN"
assert abs(float(config["eps"]) - 0.25) < 1e-6
assert int(config["min_samples"]) == 3
assert config["embedding_model"] == "all-MiniLM-L6-v2"

StatementMeta(, d19e2b24-aee5-42e5-8f75-682471521efb, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 320b557a-77d0-48df-87d2-2a01fe0c7e8d)

In [7]:
# ============================================================
# 6. Validación de GenAI
# ============================================================

success_count = (
    df_summaries
    .filter(F.col("generation_status") == "success")
    .count()
)

failed_count = (
    df_summaries
    .filter(F.col("generation_status") == "failed")
    .count()
)

print("Successful summaries:", success_count)
print("Failed summaries:", failed_count)

assert success_count == group_count
assert failed_count == 0

StatementMeta(, d19e2b24-aee5-42e5-8f75-682471521efb, 11, Finished, Available, Finished, False)

Successful summaries: 14
Failed summaries: 0


In [8]:
# ============================================================
# 7. Resultado final
# ============================================================

print("====================================")
print("FINAL PIPELINE VALIDATION: SUCCESS")
print("====================================")
print("1000 tickets processed")
print("995 batch / 5 Kafka")
print("1000 embeddings")
print("1000 clustered tickets")
print("14 clusters")
print("0 noise tickets")
print("14 incident groups")
print("14 GenAI summaries")

StatementMeta(, d19e2b24-aee5-42e5-8f75-682471521efb, 12, Finished, Available, Finished, False)

FINAL PIPELINE VALIDATION: SUCCESS
1000 tickets processed
995 batch / 5 Kafka
1000 embeddings
1000 clustered tickets
14 clusters
0 noise tickets
14 incident groups
14 GenAI summaries


In [9]:
# ============================================================
# 8. Preparación del dataset de evaluación del clustering
#
# Se combina:
#
#   Gold.TicketClusters
#       +
#   Silver.Tickets
#
# para disponer simultáneamente de:
# - cluster_id predicho
# - incident_id real
# - is_isolated
#
# incident_id se utiliza únicamente como ground truth.
# ============================================================

df_eval = (
    spark.table("Gold.TicketClusters")
    .select(
        "ticket_id",
        "cluster_id"
    )
    .join(
        spark.table("Silver.Tickets")
        .select(
            "ticket_id",
            "incident_id",
            "is_isolated"
        ),
        on="ticket_id",
        how="inner"
    )
)

eval_count = df_eval.count()

print(
    "Tickets available for clustering evaluation:",
    eval_count
)

assert eval_count == 1000

StatementMeta(, d19e2b24-aee5-42e5-8f75-682471521efb, 13, Finished, Available, Finished, False)

Tickets available for clustering evaluation: 1000


In [10]:
# ============================================================
# 9. Construcción del ground truth
#
# Los tickets recurrentes mantienen su incident_id.
#
# Cada ticket aislado recibe un identificador único para evitar
# que todos los valores null sean interpretados como un mismo
# incidente real.
# ============================================================

import pandas as pd
import numpy as np

eval_pdf = (
    df_eval
    .orderBy("ticket_id")
    .toPandas()
)


def build_ground_truth(row):

    if (
        bool(row["is_isolated"])
        or
        pd.isna(row["incident_id"])
    ):
        return (
            f"ISOLATED_{row['ticket_id']}"
        )

    return str(
        row["incident_id"]
    )


eval_pdf[
    "evaluation_incident_id"
] = (
    eval_pdf.apply(
        build_ground_truth,
        axis=1
    )
)

y_true = (
    eval_pdf[
        "evaluation_incident_id"
    ]
    .astype(str)
    .to_numpy()
)

y_pred = (
    eval_pdf[
        "cluster_id"
    ]
    .astype(int)
    .to_numpy()
)


print(
    "Ground truth groups:",
    eval_pdf[
        "evaluation_incident_id"
    ].nunique()
)

print(
    "Predicted clusters:",
    len(
        set(y_pred) - {-1}
    )
)

StatementMeta(, d19e2b24-aee5-42e5-8f75-682471521efb, 14, Finished, Available, Finished, False)

Ground truth groups: 150
Predicted clusters: 14


In [11]:
# ============================================================
# 10. Métricas globales de clustering
# ============================================================

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    homogeneity_score,
    completeness_score,
    v_measure_score
)


ari = adjusted_rand_score(
    y_true,
    y_pred
)

nmi = normalized_mutual_info_score(
    y_true,
    y_pred
)

homogeneity = homogeneity_score(
    y_true,
    y_pred
)

completeness = completeness_score(
    y_true,
    y_pred
)

v_measure = v_measure_score(
    y_true,
    y_pred
)


print("Global clustering metrics")
print("-------------------------")
print(
    "ARI:",
    ari
)
print(
    "NMI:",
    nmi
)
print(
    "Homogeneity:",
    homogeneity
)
print(
    "Completeness:",
    completeness
)
print(
    "V-measure:",
    v_measure
)

StatementMeta(, d19e2b24-aee5-42e5-8f75-682471521efb, 15, Finished, Available, Finished, False)

Global clustering metrics
-------------------------
ARI: 0.21548499022153905
NMI: 0.6874133454156105
Homogeneity: 0.5326098562827715
Completeness: 0.969076106023834
V-measure: 0.6874133454156105


In [12]:
# ============================================================
# 11. Pairwise Precision / Recall / F1
#
# Para cada pareja de tickets:
#
# Ground truth:
#   same_true = pertenecen al mismo incidente real
#
# Predicción:
#   same_pred = pertenecen al mismo cluster DBSCAN
#
# Los puntos de ruido (-1) nunca se consideran agrupados
# entre sí.
# ============================================================

def pairwise_metrics(
    y_true,
    y_pred
):
    tp = 0
    fp = 0
    fn = 0

    n = len(y_true)

    for i in range(n):

        for j in range(
            i + 1,
            n
        ):

            same_true = (
                y_true[i]
                ==
                y_true[j]
            )

            same_pred = (
                y_pred[i] != -1
                and
                y_pred[j] != -1
                and
                y_pred[i]
                ==
                y_pred[j]
            )

            if (
                same_true
                and
                same_pred
            ):
                tp += 1

            elif (
                not same_true
                and
                same_pred
            ):
                fp += 1

            elif (
                same_true
                and
                not same_pred
            ):
                fn += 1


    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    f1 = (
        2 * precision * recall
        /
        (precision + recall)
        if (
            precision + recall
        ) > 0
        else 0
    )


    return (
        precision,
        recall,
        f1,
        tp,
        fp,
        fn
    )

StatementMeta(, d19e2b24-aee5-42e5-8f75-682471521efb, 16, Finished, Available, Finished, False)

In [13]:
# ============================================================
# 12. Evaluación pairwise
# ============================================================

(
    pairwise_precision,
    pairwise_recall,
    pairwise_f1,
    pairwise_tp,
    pairwise_fp,
    pairwise_fn
) = pairwise_metrics(
    y_true,
    y_pred
)


print("Pairwise clustering metrics")
print("---------------------------")

print(
    "Precision:",
    pairwise_precision
)

print(
    "Recall:",
    pairwise_recall
)

print(
    "F1:",
    pairwise_f1
)

print(
    "TP:",
    pairwise_tp
)

print(
    "FP:",
    pairwise_fp
)

print(
    "FN:",
    pairwise_fn
)

StatementMeta(, d19e2b24-aee5-42e5-8f75-682471521efb, 17, Finished, Available, Finished, False)

Pairwise clustering metrics
---------------------------
Precision: 0.13646092845362995
Recall: 0.9373342974210653
F1: 0.2382381769174222
TP: 7778
FP: 49220
FN: 520


In [14]:
# ============================================================
# 13. Análisis de clusters heterogéneos
#
# Un cluster se considera "merged" cuando contiene tickets
# procedentes de más de un incidente real.
#
# Para este análisis excluimos los tickets aislados, ya que
# cada uno representa por definición un caso independiente.
# ============================================================

df_ground_truth_groups = (
    df_eval
    .filter(
        F.col(
            "incident_id"
        ).isNotNull()
    )
    .groupBy(
        "cluster_id"
    )
    .agg(
        F.countDistinct(
            "incident_id"
        )
        .alias(
            "ground_truth_incident_count"
        )
    )
)

display(
    df_ground_truth_groups
    .orderBy(
        F.desc(
            "ground_truth_incident_count"
        )
    )
)

StatementMeta(, d19e2b24-aee5-42e5-8f75-682471521efb, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3b0277af-3bf4-48f8-abdc-c928c62bf52f)

In [15]:
# ============================================================
# 14. Porcentaje de grupos mezclados
# ============================================================

real_groups_count = (
    df_ground_truth_groups
    .count()
)

merged_groups = (
    df_ground_truth_groups
    .filter(
        F.col(
            "ground_truth_incident_count"
        ) > 1
    )
    .count()
)

merged_percentage = (
    merged_groups
    /
    real_groups_count
    * 100
    if real_groups_count > 0
    else 0
)


print("Cluster purity analysis")
print("-----------------------")

print(
    "Groups analysed:",
    real_groups_count
)

print(
    "Merged groups:",
    merged_groups
)

print(
    "Merged percentage:",
    round(
        merged_percentage,
        2
    ),
    "%"
)

StatementMeta(, d19e2b24-aee5-42e5-8f75-682471521efb, 19, Finished, Available, Finished, False)

Cluster purity analysis
-----------------------
Groups analysed: 9
Merged groups: 9
Merged percentage: 100.0 %


In [16]:
# ============================================================
# 15. Resumen de evaluación
# ============================================================

evaluation_summary = pd.DataFrame([
    {
        "metric":
            "ARI",

        "value":
            ari
    },
    {
        "metric":
            "NMI",

        "value":
            nmi
    },
    {
        "metric":
            "Pairwise Precision",

        "value":
            pairwise_precision
    },
    {
        "metric":
            "Pairwise Recall",

        "value":
            pairwise_recall
    },
    {
        "metric":
            "Pairwise F1",

        "value":
            pairwise_f1
    },
    {
        "metric":
            "Merged Groups Ratio",

        "value":
            merged_percentage / 100
    }
])


display(
    evaluation_summary
)

StatementMeta(, d19e2b24-aee5-42e5-8f75-682471521efb, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2e688472-c82c-4813-a197-bf02a5b44f42)

In [17]:
# ============================================================
# 16. Validación final de regresión
# ============================================================

print(
    "========================================"
)

print(
    "FINAL PIPELINE REGRESSION TEST: SUCCESS"
)

print(
    "========================================"
)

print()

print(
    "Functional validation"
)

print(
    "---------------------"
)

print(
    f"Silver tickets: {silver_count}"
)

print(
    f"Embeddings: {embedding_count}"
)

print(
    f"Clustered tickets: {clustered_count}"
)

print(
    f"Incident groups: {group_count}"
)

print(
    f"GenAI summaries: {summary_count}"
)

print()

print(
    "Clustering validation"
)

print(
    "---------------------"
)

print(
    f"ARI: {ari:.4f}"
)

print(
    f"NMI: {nmi:.4f}"
)

print(
    "Pairwise Precision:",
    f"{pairwise_precision:.4f}"
)

print(
    "Pairwise Recall:",
    f"{pairwise_recall:.4f}"
)

print(
    "Pairwise F1:",
    f"{pairwise_f1:.4f}"
)

print(
    "Merged groups:",
    f"{merged_groups}/{real_groups_count}",
    f"({merged_percentage:.2f}%)"
)

print()

print(
    "The refactored operational pipeline "
    "produces the expected functional and "
    "clustering results."
)

StatementMeta(, d19e2b24-aee5-42e5-8f75-682471521efb, 21, Finished, Available, Finished, False)

FINAL PIPELINE REGRESSION TEST: SUCCESS

Functional validation
---------------------
Silver tickets: 1000
Embeddings: 1000
Clustered tickets: 1000
Incident groups: 14
GenAI summaries: 14

Clustering validation
---------------------
ARI: 0.2155
NMI: 0.6874
Pairwise Precision: 0.1365
Pairwise Recall: 0.9373
Pairwise F1: 0.2382
Merged groups: 9/9 (100.00%)

The refactored operational pipeline produces the expected functional and clustering results.
